# 06_panderm_linear_probe_ham

This notebook does a **quick PanDerm Base linear-probe experiment on HAM**.

## Purpose

This is **not** the final thesis model.  
It is a **screening experiment** to answer:

> Are PanDerm Base embeddings already strong enough on the HAM classification task to justify more work such as fine-tuning?

## Why linear probing first?

PanDerm Base is a **foundation encoder**.  
A linear probe lets you:

- keep the encoder **frozen**
- extract embeddings once
- train a simple linear classifier on top
- get a fast sense of feature quality

This is a good decision step before:
- full fine-tuning on the GPU cluster
- integrating PanDerm into the CAM pipeline

## Data used here

This notebook uses the existing HAM splits:
- `train_base.csv`
- `val_base.csv`

So this is a **HAM-only quick benchmark**, not the full HAM→BCN transfer setup.

## Outputs

Saved to `../outputs/panderm_linear_probe_ham/`:
- extracted embeddings
- validation predictions
- metrics json
- confusion matrix figure
- PCA projection figure


## What to look for at the end

At the end of this notebook, ask:

1. Is PanDerm linear probing already competitive with the ConvNeXt baseline?
2. Are difficult classes still weak?
3. Does the confusion structure look promising?
4. Is it worth moving to PanDerm fine-tuning next?

If yes, the next step is:
- **PanDerm fine-tuning on cluster**
- then later **CAM / Finer-CAM**


In [12]:
from pathlib import Path
import json
import time
import random
import sys

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    recall_score,
)
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt


## Config

Edit the paths below if needed.

This notebook assumes it is placed in:
- `notebooks/06_panderm_linear_probe_ham.ipynb`

and that your repo structure matches your current setup.


In [14]:
TRAIN_CSV = Path("../data/processed/splits/train_base.csv")
VAL_CSV = Path("../data/processed/splits/val_base.csv")
PANDERM_CKPT = Path("../external/weights/panderm_bb_data6_checkpoint-499.pth")

OUT_DIR = Path("../outputs/panderm_linear_probe_ham")
EMB_DIR = OUT_DIR / "embeddings"
METRICS_DIR = OUT_DIR / "metrics"
FIG_DIR = OUT_DIR / "figures"
PRED_DIR = OUT_DIR / "predictions"

for p in [OUT_DIR, EMB_DIR, METRICS_DIR, FIG_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 0

PRIMARY_CLASSES = ["NV", "MEL", "BCC", "BKL", "AKIEC", "DF"]
CLASS_TO_IDX = {c: i for i, c in enumerate(PRIMARY_CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

CONVNEXT_HAM_VAL_BAL_ACC = 0.5421862474408425

MAX_ITER = 1000
C_VALUE = 1.0
CLASS_WEIGHT = "balanced"

if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("DEVICE:", DEVICE)
print("TRAIN_CSV exists:", TRAIN_CSV.exists(), TRAIN_CSV)
print("VAL_CSV exists:", VAL_CSV.exists(), VAL_CSV)
print("PANDERM_CKPT exists:", PANDERM_CKPT.exists(), PANDERM_CKPT)
print("OUT_DIR:", OUT_DIR.resolve())

# Change this path if needed so Python can import the PanDerm repo code
PANDERM_CLASSIFICATION_DIR = Path("../external/PanDerm/classification").resolve()
if str(PANDERM_CLASSIFICATION_DIR) not in sys.path:
    sys.path.insert(0, str(PANDERM_CLASSIFICATION_DIR))

from models.modeling_finetune import panderm_base_patch16_224
from models.builder import get_eval_transforms


DEVICE: mps
TRAIN_CSV exists: True ../data/processed/splits/train_base.csv
VAL_CSV exists: True ../data/processed/splits/val_base.csv
PANDERM_CKPT exists: True ../external/weights/panderm_bb_data6_checkpoint-499.pth
OUT_DIR: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/panderm_linear_probe_ham


## Reproducibility


In [15]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(SEED)
print("Seeded with:", SEED)


Seeded with: 42


## Load HAM train / val splits


In [16]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print("train_df:", train_df.shape)
print("val_df:", val_df.shape)

print("\nTrain label counts:")
display(train_df["harmonized_label"].value_counts().reindex(PRIMARY_CLASSES, fill_value=0).to_frame("count"))

print("\nVal label counts:")
display(val_df["harmonized_label"].value_counts().reindex(PRIMARY_CLASSES, fill_value=0).to_frame("count"))


train_df: (9052, 32)
val_df: (2258, 32)

Train label counts:


,count
harmonized_label,
NV,6195
MEL,1054
BCC,494
BKL,1061
AKIEC,120
DF,128



Val label counts:


,count
harmonized_label,
NV,1541
MEL,251
BCC,128
BKL,277
AKIEC,29
DF,32


## Dataset and transforms


In [17]:
def safe_open_image(path):
    path = Path(path)
    with Image.open(path) as img:
        return img.convert("RGB")

# eval_transform = T.Compose([
#     T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
#     T.ToTensor(),
#     T.Normalize(mean=[0.485, 0.456, 0.406],
#                 std=[0.229, 0.224, 0.225]),
# ])

eval_transform = get_eval_transforms(
    which_img_norm="imagenet",
    img_resize=256,
    center_crop=True
)

class SkinCSVEmbeddingDataset(Dataset):
    def __init__(self, df, class_to_idx, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = safe_open_image(row["image_path"])
        if self.transform is not None:
            img = self.transform(img)

        y = self.class_to_idx[row["harmonized_label"]]
        meta = {
            "isic_id": row["isic_id"],
            "lesion_id": row["lesion_id"],
            "image_path": row["image_path"],
            "label_str": row["harmonized_label"],
        }
        return img, y, meta

train_ds = SkinCSVEmbeddingDataset(train_df, CLASS_TO_IDX, transform=eval_transform)
val_ds = SkinCSVEmbeddingDataset(val_df, CLASS_TO_IDX, transform=eval_transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))


normalization method:  imagenet
train batches: 142
val batches: 36


## Build PanDerm Base encoder

We use a **ViT-B/16** backbone and load the PanDerm Base checkpoint.

This notebook uses the encoder as a **feature extractor** only.


In [18]:
def build_panderm_base_encoder():
    # This is the actual PanDerm Base architecture
    model = panderm_base_patch16_224()
    return model


def load_panderm_checkpoint(model, checkpoint_path, device="cpu"):
    checkpoint_path = Path(checkpoint_path)
    print(f"Loading checkpoint from: {checkpoint_path}")

    # builder.py uses exactly this style for PanDerm_Base_LP
    state_dict = torch.load(checkpoint_path, map_location="cpu")
    missing, unexpected = model.load_state_dict(state_dict, strict=False)

    print("=" * 80)
    print("PanDerm checkpoint loading report")
    print("=" * 80)
    print("Missing keys:", len(missing))
    if len(missing) > 0:
        print("First missing keys:", missing[:20])
    print("Unexpected keys:", len(unexpected))
    if len(unexpected) > 0:
        print("First unexpected keys:", unexpected[:20])

    return model


encoder = build_panderm_base_encoder()
encoder = load_panderm_checkpoint(encoder, PANDERM_CKPT, device=DEVICE)

# Same as builder.py
encoder.head = torch.nn.Identity()

encoder = encoder.to(DEVICE)
encoder.eval()

n_params = sum(p.numel() for p in encoder.parameters())
print("Encoder params:", f"{n_params:,}")

/Users/choekyelnyungmartsang/opt/anaconda3/envs/thesis/lib/python3.11/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1711403251597/work/aten/src/ATen/native/TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Loading checkpoint from: ../external/weights/panderm_bb_data6_checkpoint-499.pth
PanDerm checkpoint loading report
Missing keys: 2
First missing keys: ['head.weight', 'head.bias']
Unexpected keys: 0
Encoder params: 85,807,872


## Extract embeddings with tqdm

This is the main part where time matters on laptop.  
The progress bars help you estimate how long extraction takes.


In [19]:
@torch.no_grad()
def extract_embeddings(model, loader, device, split_name="train"):
    model.eval()

    all_embeddings = []
    all_labels = []
    all_rows = []

    start = time.time()

    pbar = tqdm(loader, desc=f"extract_{split_name}", leave=True)
    for imgs, labels, metas in pbar:
        imgs = imgs.to(device, non_blocking=(device == "cuda"))

        feats = model(imgs)
        feats = feats.detach().cpu().numpy()

        all_embeddings.append(feats)
        all_labels.append(labels.numpy())

        batch_size = len(labels)
        for i in range(batch_size):
            row = {k: metas[k][i] for k in metas}
            row["label_idx"] = int(labels[i].item())
            all_rows.append(row)

    elapsed = time.time() - start

    embeddings = np.concatenate(all_embeddings, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    meta_df = pd.DataFrame(all_rows)

    print(f"Finished {split_name} extraction in {elapsed:.2f} sec")
    print(f"{split_name} embeddings shape:", embeddings.shape)

    return embeddings, labels, meta_df, elapsed

train_emb, train_y, train_meta, train_time = extract_embeddings(encoder, train_loader, DEVICE, split_name="train")
val_emb, val_y, val_meta, val_time = extract_embeddings(encoder, val_loader, DEVICE, split_name="val")

np.save(EMB_DIR / "train_embeddings.npy", train_emb)
np.save(EMB_DIR / "val_embeddings.npy", val_emb)
np.save(EMB_DIR / "train_labels.npy", train_y)
np.save(EMB_DIR / "val_labels.npy", val_y)
train_meta.to_csv(EMB_DIR / "train_meta.csv", index=False)
val_meta.to_csv(EMB_DIR / "val_meta.csv", index=False)

print("Saved embeddings and metadata to:", EMB_DIR.resolve())


extract_train:   0%|          | 0/142 [00:00<?, ?it/s]

Finished train extraction in 154.97 sec
train embeddings shape: (9052, 768)


extract_val:   0%|          | 0/36 [00:00<?, ?it/s]

Finished val extraction in 37.84 sec
val embeddings shape: (2258, 768)
Saved embeddings and metadata to: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/panderm_linear_probe_ham/embeddings


## Fit a linear probe

We use multinomial logistic regression on frozen PanDerm embeddings.

This is the classic linear-probing idea:
- encoder fixed
- only linear classifier learned


In [20]:
probe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=MAX_ITER,
        C=C_VALUE,
        class_weight=CLASS_WEIGHT,
        # multi_class="multinomial",
        solver="lbfgs",
        random_state=SEED,
    ))
])

fit_start = time.time()
probe.fit(train_emb, train_y)
fit_time = time.time() - fit_start

print(f"Linear probe fit finished in {fit_time:.2f} sec")


Linear probe fit finished in 6.48 sec


## Validation predictions


In [21]:
val_pred = probe.predict(val_emb)
val_prob = probe.predict_proba(val_emb)

pred_df = val_meta.copy()
pred_df["y_true_idx"] = val_y
pred_df["y_true"] = [IDX_TO_CLASS[i] for i in val_y]
pred_df["y_pred_idx"] = val_pred
pred_df["y_pred"] = [IDX_TO_CLASS[i] for i in val_pred]
pred_df["correct"] = pred_df["y_true_idx"] == pred_df["y_pred_idx"]

for i, cls in IDX_TO_CLASS.items():
    pred_df[f"prob_{cls}"] = val_prob[:, i]

pred_df.to_csv(PRED_DIR / "ham_val_predictions_linear_probe.csv", index=False)
print("Saved prediction CSV:", (PRED_DIR / "ham_val_predictions_linear_probe.csv").resolve())

display(pred_df.head())


Saved prediction CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/panderm_linear_probe_ham/predictions/ham_val_predictions_linear_probe.csv


,isic_id,lesion_id,image_path,label_str,label_idx,y_true_idx,y_true,y_pred_idx,y_pred,correct,prob_NV,prob_MEL,prob_BCC,prob_BKL,prob_AKIEC,prob_DF
0,ISIC_0024311,IL_9035157,../data/HAM10k/ISIC_0024311.jpg,NV,0,0,NV,0,NV,True,9.999952e-01,4.755932e-06,3.048706e-20,2.767854e-09,2.878240e-12,5.056599e-12
1,ISIC_0024318,IL_9624462,../data/HAM10k/ISIC_0024318.jpg,DF,5,5,DF,5,DF,True,3.671616e-10,1.672300e-11,3.877154e-11,4.750436e-07,1.006252e-13,9.999995e-01
2,ISIC_0024321,IL_4111475,../data/HAM10k/ISIC_0024321.jpg,NV,0,0,NV,0,NV,True,7.144126e-01,5.983921e-06,1.376992e-07,7.456107e-11,1.487790e-12,2.855813e-01
3,ISIC_0024330,IL_0184125,../data/HAM10k/ISIC_0024330.jpg,DF,5,5,DF,5,DF,True,2.456490e-04,3.993274e-03,9.039981e-03,4.403488e-04,1.896205e-07,9.862806e-01
4,ISIC_0024333,IL_3589188,../data/HAM10k/ISIC_0024333.jpg,MEL,1,1,MEL,3,BKL,False,1.476083e-01,3.185225e-02,1.110841e-11,8.189471e-01,4.750746e-09,1.592377e-03


## Metrics


In [22]:
def compute_metrics(y_true, y_pred, class_names):
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    per_class_recall = recall_score(
        y_true, y_pred,
        average=None,
        labels=list(range(len(class_names))),
        zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))

    return {
        "accuracy": float(acc),
        "balanced_accuracy": float(bal_acc),
        "macro_f1": float(macro_f1),
        "per_class_recall": {class_names[i]: float(per_class_recall[i]) for i in range(len(class_names))},
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(
            y_true, y_pred,
            labels=list(range(len(class_names))),
            target_names=class_names,
            output_dict=True,
            zero_division=0
        )
    }

metrics = compute_metrics(val_y, val_pred, PRIMARY_CLASSES)
metrics["train_extraction_seconds"] = float(train_time)
metrics["val_extraction_seconds"] = float(val_time)
metrics["probe_fit_seconds"] = float(fit_time)

with open(METRICS_DIR / "ham_val_linear_probe_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(json.dumps({
    "accuracy": metrics["accuracy"],
    "balanced_accuracy": metrics["balanced_accuracy"],
    "macro_f1": metrics["macro_f1"],
    "per_class_recall": metrics["per_class_recall"],
    "convnext_ham_val_bal_acc_reference": CONVNEXT_HAM_VAL_BAL_ACC,
}, indent=2))


{
  "accuracy": 0.8051372896368467,
  "balanced_accuracy": 0.7005978989201793,
  "macro_f1": 0.6598284517081503,
  "per_class_recall": {
    "NV": 0.8617780661907852,
    "MEL": 0.6215139442231076,
    "BCC": 0.8515625,
    "BKL": 0.6823104693140795,
    "AKIEC": 0.6551724137931034,
    "DF": 0.53125
  },
  "convnext_ham_val_bal_acc_reference": 0.5421862474408425
}


## Compare against your ConvNeXt baseline


In [23]:
delta_bal_acc = metrics["balanced_accuracy"] - CONVNEXT_HAM_VAL_BAL_ACC

print("PanDerm linear probe HAM val balanced accuracy:", round(metrics["balanced_accuracy"], 4))
print("ConvNeXt HAM val balanced accuracy (reference):", round(CONVNEXT_HAM_VAL_BAL_ACC, 4))
print("Difference:", round(delta_bal_acc, 4))

if delta_bal_acc > 0.03:
    print("\nInterpretation: PanDerm linear probing looks clearly stronger than the current ConvNeXt baseline.")
elif delta_bal_acc > -0.02:
    print("\nInterpretation: PanDerm linear probing is in the same ballpark as the current ConvNeXt baseline.")
else:
    print("\nInterpretation: PanDerm linear probing is clearly weaker than the current ConvNeXt baseline.")


PanDerm linear probe HAM val balanced accuracy: 0.7006
ConvNeXt HAM val balanced accuracy (reference): 0.5422
Difference: 0.1584

Interpretation: PanDerm linear probing looks clearly stronger than the current ConvNeXt baseline.


## Confusion matrix


In [ ]:
cm = np.array(metrics["confusion_matrix"])

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111)
ax.imshow(cm, interpolation="nearest")
ax.set_title("HAM val confusion matrix: PanDerm linear probe")
ax.set_xticks(range(len(PRIMARY_CLASSES)))
ax.set_yticks(range(len(PRIMARY_CLASSES)))
ax.set_xticklabels(PRIMARY_CLASSES, rotation=45, ha="right")
ax.set_yticklabels(PRIMARY_CLASSES)
ax.set_ylabel("True")
ax.set_xlabel("Predicted")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.tight_layout()
plt.savefig(FIG_DIR / "ham_val_confusion_matrix_linear_probe.png", dpi=200, bbox_inches="tight")
plt.show()

print("Saved confusion matrix figure:", (FIG_DIR / "ham_val_confusion_matrix_linear_probe.png").resolve())


## Top confusion pairs


In [25]:
confusions = (
    pred_df.loc[~pred_df["correct"]]
    .groupby(["y_true", "y_pred"])
    .size()
    .sort_values(ascending=False)
    .to_frame("count")
    .reset_index()
)

confusions.to_csv(METRICS_DIR / "ham_val_top_confusions_linear_probe.csv", index=False)
display(confusions.head(20))


,y_true,y_pred,count
0,NV,MEL,126
1,NV,BKL,60
2,MEL,BKL,41
3,MEL,NV,38
4,BKL,MEL,36
5,BKL,NV,28
6,NV,BCC,22
7,BKL,BCC,12
8,BKL,AKIEC,10
9,MEL,BCC,10


## PCA visualization of validation embeddings

This is only a rough qualitative view, but it can help you see:
- whether classes form separable regions
- which classes overlap strongly


In [ ]:
pca = PCA(n_components=2, random_state=SEED)
val_pca = pca.fit_transform(val_emb)

pca_df = pred_df.copy()
pca_df["pc1"] = val_pca[:, 0]
pca_df["pc2"] = val_pca[:, 1]

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)

for cls in PRIMARY_CLASSES:
    sub = pca_df[pca_df["y_true"] == cls]
    ax.scatter(sub["pc1"], sub["pc2"], s=12, alpha=0.6, label=cls)

ax.set_title("PCA of HAM val PanDerm embeddings")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "ham_val_pca_linear_probe.png", dpi=200, bbox_inches="tight")
plt.show()

print("Saved PCA figure:", (FIG_DIR / "ham_val_pca_linear_probe.png").resolve())


## Quick inspection subsets for next steps

These CSVs are useful if you want to later:
- inspect image examples manually
- choose cases for CAM generation
- compare PanDerm against ConvNeXt on the same difficult class pairs


In [27]:
subset_specs = [
    ("correct_mel", (pred_df["y_true"] == "MEL") & (pred_df["y_pred"] == "MEL"), "prob_MEL"),
    ("correct_nv", (pred_df["y_true"] == "NV") & (pred_df["y_pred"] == "NV"), "prob_NV"),
    ("mel_as_nv", (pred_df["y_true"] == "MEL") & (pred_df["y_pred"] == "NV"), "prob_NV"),
    ("nv_as_mel", (pred_df["y_true"] == "NV") & (pred_df["y_pred"] == "MEL"), "prob_MEL"),
]

review_dir = OUT_DIR / "review_sets"
review_dir.mkdir(parents=True, exist_ok=True)

for name, mask, sort_col in subset_specs:
    sub = pred_df.loc[mask].sort_values(sort_col, ascending=False).copy()
    sub.to_csv(review_dir / f"{name}.csv", index=False)
    print(name, "->", len(sub), "rows")


correct_mel -> 156 rows
correct_nv -> 1328 rows
mel_as_nv -> 38 rows
nv_as_mel -> 126 rows


## Suggested interpretation

Use the following logic:

### If PanDerm linear probing is clearly stronger than ConvNeXt
Next step:
- move to **PanDerm fine-tuning on cluster**

### If PanDerm linear probing is similar to ConvNeXt
Next step:
- PanDerm is still promising, especially because it is frozen here
- fine-tuning may still be worthwhile

### If PanDerm linear probing is clearly weaker
Next step:
- either stay with ConvNeXt
- or revisit checkpoint loading / architecture compatibility


In [28]:
print("=" * 80)
print("NEXT-STEP SUMMARY")
print("=" * 80)

bal_acc = metrics["balanced_accuracy"]

if bal_acc > CONVNEXT_HAM_VAL_BAL_ACC + 0.03:
    print("1. PanDerm linear probing looks clearly promising.")
    print("2. Recommended next step: run PanDerm fine-tuning on cluster.")
    print("3. After that, compare confusion pairs and then generate CAMs.")
elif bal_acc > CONVNEXT_HAM_VAL_BAL_ACC - 0.02:
    print("1. PanDerm linear probing is in the same ballpark as ConvNeXt.")
    print("2. Recommended next step: PanDerm fine-tuning is still worth trying.")
    print("3. Fine-tuning may unlock stronger task-specific performance than frozen features.")
else:
    print("1. PanDerm linear probing is weaker than expected.")
    print("2. Before fine-tuning, check checkpoint loading compatibility carefully.")
    print("3. Also compare confusion structure, not only one scalar metric.")

print("\nMost important files created:")
print("-", METRICS_DIR / "ham_val_linear_probe_metrics.json")
print("-", METRICS_DIR / "ham_val_top_confusions_linear_probe.csv")
print("-", PRED_DIR / "ham_val_predictions_linear_probe.csv")
print("-", FIG_DIR / "ham_val_confusion_matrix_linear_probe.png")
print("-", FIG_DIR / "ham_val_pca_linear_probe.png")


NEXT-STEP SUMMARY
1. PanDerm linear probing looks clearly promising.
2. Recommended next step: run PanDerm fine-tuning on cluster.
3. After that, compare confusion pairs and then generate CAMs.

Most important files created:
- ../outputs/panderm_linear_probe_ham/metrics/ham_val_linear_probe_metrics.json
- ../outputs/panderm_linear_probe_ham/metrics/ham_val_top_confusions_linear_probe.csv
- ../outputs/panderm_linear_probe_ham/predictions/ham_val_predictions_linear_probe.csv
- ../outputs/panderm_linear_probe_ham/figures/ham_val_confusion_matrix_linear_probe.png
- ../outputs/panderm_linear_probe_ham/figures/ham_val_pca_linear_probe.png


In [ ]:
from efficientnet_pytorch import EfficientNet

CKPT_PATH = Path("../external/weights/isic7_last_effnetb4.pth")

hf_class_to_idx = {"AK": 0, "BCC": 1, "BKL": 2, "DF": 3, "MEL": 4, "NV": 5, "VASC": 6}
hf_idx_to_class = {v: k for k, v in hf_class_to_idx.items()}

hf_to_primary = {
    "AK": "AKIEC",
    "BCC": "BCC",
    "BKL": "BKL",
    "DF": "DF",
    "MEL": "MEL",
    "NV": "NV",
}

effnet = EfficientNet.from_name("efficientnet-b4")
effnet._fc = torch.nn.Linear(effnet._fc.in_features, 7)

state = torch.load(CKPT_PATH, map_location="cpu")
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]

cleaned = {}
for k, v in state.items():
    if k.startswith("module."):
        k = k[len("module."):]
    cleaned[k] = v

missing, unexpected = effnet.load_state_dict(cleaned, strict=False)
print("Missing:", len(missing))
print("Unexpected:", len(unexpected))
if missing:
    print("Missing sample:", missing[:10])
if unexpected:
    print("Unexpected sample:", unexpected[:10])

effnet = effnet.to(DEVICE)
effnet.eval()

In [30]:
effnet_transform = T.Compose([
    T.Resize((380, 380)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

class SkinCSVClassificationDatasetHF(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = safe_open_image(row["image_path"])
        if self.transform is not None:
            img = self.transform(img)

        meta = {
            "isic_id": row["isic_id"],
            "lesion_id": row["lesion_id"],
            "image_path": row["image_path"],
            "y_true": row["harmonized_label"],
        }
        return img, meta

hf_val_ds = SkinCSVClassificationDatasetHF(val_df, transform=effnet_transform)
hf_val_loader = DataLoader(
    hf_val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
)

In [31]:
@torch.no_grad()
def predict_effnet_ham_val(model, loader, device):
    rows = []
    softmax = torch.nn.Softmax(dim=1)

    for imgs, metas in tqdm(loader, desc="effnet_val", leave=True):
        imgs = imgs.to(device, non_blocking=(device == "cuda"))
        logits = model(imgs)
        probs = softmax(logits)
        preds = logits.argmax(dim=1).cpu().numpy()

        batch_size = imgs.size(0)
        for i in range(batch_size):
            hf_pred = hf_idx_to_class[int(preds[i])]
            primary_pred = hf_to_primary.get(hf_pred, None)

            row = {k: metas[k][i] for k in metas}
            row["hf_pred"] = hf_pred
            row["y_pred"] = primary_pred

            for j, cls in hf_idx_to_class.items():
                row[f"prob_{cls}"] = float(probs[i, j].item())

            rows.append(row)

    return pd.DataFrame(rows)

effnet_pred_df = predict_effnet_ham_val(effnet, hf_val_loader, DEVICE)
display(effnet_pred_df.head())

effnet_val:   0%|          | 0/71 [00:00<?, ?it/s]

,isic_id,lesion_id,image_path,y_true,hf_pred,y_pred,prob_AK,prob_BCC,prob_BKL,prob_DF,prob_MEL,prob_NV,prob_VASC
0,ISIC_0024311,IL_9035157,../data/HAM10k/ISIC_0024311.jpg,NV,NV,NV,2.758341e-15,4.333999e-13,4.870700e-13,8.713255e-13,2.998082e-15,1.000000e+00,2.662055e-13
1,ISIC_0024318,IL_9624462,../data/HAM10k/ISIC_0024318.jpg,DF,DF,DF,0.000000e+00,0.000000e+00,1.018379e-35,1.000000e+00,0.000000e+00,1.141322e-37,8.168986e-24
2,ISIC_0024321,IL_4111475,../data/HAM10k/ISIC_0024321.jpg,NV,NV,NV,6.036108e-16,7.801068e-14,4.331627e-13,3.448296e-12,9.147311e-13,1.000000e+00,2.480537e-13
3,ISIC_0024330,IL_0184125,../data/HAM10k/ISIC_0024330.jpg,DF,DF,DF,3.213290e-34,3.850422e-37,4.957627e-34,1.000000e+00,1.077866e-35,4.111070e-33,1.226277e-24
4,ISIC_0024333,IL_3589188,../data/HAM10k/ISIC_0024333.jpg,MEL,MEL,MEL,1.358660e-16,3.260039e-16,2.600334e-13,1.210020e-12,1.000000e+00,3.435427e-13,3.541417e-14


In [32]:
effnet_eval_df = effnet_pred_df[effnet_pred_df["y_pred"].notna()].copy()

y_true = effnet_eval_df["y_true"].map(CLASS_TO_IDX).values
y_pred = effnet_eval_df["y_pred"].map(CLASS_TO_IDX).values

effnet_metrics = compute_metrics(y_true, y_pred, PRIMARY_CLASSES)

print(json.dumps({
    "accuracy": effnet_metrics["accuracy"],
    "balanced_accuracy": effnet_metrics["balanced_accuracy"],
    "macro_f1": effnet_metrics["macro_f1"],
    "per_class_recall": effnet_metrics["per_class_recall"],
}, indent=2))

{
  "accuracy": 0.9631765749778172,
  "balanced_accuracy": 0.9418821486249612,
  "macro_f1": 0.9433960392903863,
  "per_class_recall": {
    "NV": 0.9811810512654121,
    "MEL": 0.9236947791164659,
    "BCC": 0.9365079365079365,
    "BKL": 0.9133574007220217,
    "AKIEC": 0.896551724137931,
    "DF": 1.0
  }
}


In [33]:
comparison_df = pd.DataFrame([
    {
        "model": "HF EfficientNet-B4",
        "accuracy": effnet_metrics["accuracy"],
        "balanced_accuracy": effnet_metrics["balanced_accuracy"],
        "macro_f1": effnet_metrics["macro_f1"],
    },
    {
        "model": "PanDerm Base linear probe",
        "accuracy": metrics["accuracy"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "macro_f1": metrics["macro_f1"],
    },
])

display(comparison_df)

,model,accuracy,balanced_accuracy,macro_f1
0,HF EfficientNet-B4,0.963177,0.941882,0.943396
1,PanDerm Base linear probe,0.805137,0.700598,0.659828
